# Day 2 — automatic backward via topo-sort

Yesterday you wrote `_backward` closures by hand for a specific graph. Today we generalize: any graph, one `.backward()` call.

## What you ship today

1. `Value.backward()` that walks the graph in reverse topological order and calls each node's `_backward` closure.
2. Operators: `exp`, `**` (scalar power), `/`, `relu`, `__neg__`, `__sub__`, `__radd__`, `__rmul__`.
3. Gradient-check 5 random expressions against PyTorch.

## The single most important thing

The `_backward` for each op is **just the local derivative**, multiplied by the *output* node's already-accumulated `.grad`. The chain rule is the multiplication; the topo order ensures the output's grad is finished before any of its inputs are visited.

## Common traps (re-stating from README.md)

1. Accumulate gradients with `+=`, never assign. (Bites you when a `Value` is reused.)
2. Topo order is **post-order DFS, then reverse**. Pre-order will mostly work and silently break on diamonds.
3. `__radd__` exists for things like `2 + v`. One-liner, easy to forget.

In [ ]:
# Reference topo sort. Don't paste blindly — type it.
def build_topo(root):
    topo = []
    visited = set()
    def visit(v):
        if v in visited: return
        visited.add(v)
        for parent in v._prev:
            visit(parent)
        topo.append(v)
    visit(root)
    return topo  # reverse this for backward order

## Each op's local derivative — derive each before coding

| op            | forward            | local d/d(self)             | local d/d(other) |
|---------------|--------------------|-----------------------------|------------------|
| add           | self + other       | 1                           | 1                |
| mul           | self * other       | other                       | self             |
| pow (int n)   | self ** n          | n * self**(n-1)             | n/a              |
| exp           | exp(self)          | exp(self) (= output)        | n/a              |
| tanh          | tanh(self)         | 1 - tanh(self)**2 (= 1 - output**2) | n/a       |
| relu          | max(0, self)       | 1 if self > 0 else 0        | n/a              |

Then write `_backward` for each as: `self.grad += <local d/d(self)> * out.grad`.

In [ ]:
# Gradient check harness: random expression, your Value vs torch.
import random, torch

def random_expr(a_val, b_val, c_val):
    # Returns a string and a function f(a, b, c) that builds it.
    # Examples:
    #   ((a + b) * c).tanh()
    #   (a * b - c).exp()
    #   (a**2 + (b * c).relu())
    # TODO: define 5 of these by hand.
    pass

def grad_check(f_micro, f_torch, a, b, c, atol=1e-5):
    # Build both, run backward, compare .grad on each input.
    pass

## End-of-day check

- [ ] `Value.backward()` works on any graph (test with diamond: `b = a + a; c = b * b; c.backward()`).
- [ ] All 5 random gradient-check expressions agree with PyTorch to atol=1e-5.
- [ ] You can articulate why the topo order needs to be reversed before the backward sweep.

Append surprise/insight to `NOTES.md`.